# Experiment 3.3: adaptive frontier-resolution fine-tuning

Experiment 3.3 is a new experiment identity initialized from the strongest Experiment 3.2 policy:

```text
Experiment 3.2
    best learned exploration policy
            |
            v
Experiment 3.3
    same policy-weight initialization
    + adaptive frontier-resolution reward
    + fresh target, optimizer, replay, metrics, and output directory
```

The notebook is intentionally thin. Environment interaction, rewards, training, validation, checkpoint selection, GIF diagnostics, and metrics remain in the reusable `utils/minigrid_exploration_*` framework. Experiment 3.2 is never written or modified here.

## Imports

In [ ]:
from dataclasses import asdict, replace
from pathlib import Path
import json

import numpy as np
import torch

from utils import alert_training_complete
from utils.minigrid_exploration import (
    CheckpointManager,
    ContinuationConfig,
    MiniGridExplorationEvaluator,
    MiniGridExplorationExperimentConfig,
    MiniGridExplorationTrainer,
    OutputConfig,
    compute_adaptive_frontier_resolution_reward,
    save_comparison_artifacts,
)
from utils.minigrid_exploration_rewards import ExplorationRewardConfig

## Experiment 3.3 controls

Only `RUN_EXPERIMENT_3_3` needs to change to launch training after the parent best checkpoint exists. Validation and best-checkpoint tracking run automatically according to the inherited Experiment 3.2 schedule.

In [ ]:
EXPERIMENT_NAME = "experiment_3_3"
INITIALIZE_FROM_EXPERIMENT = "experiment_3_2"
INITIALIZE_FROM = "best"
TRAINING_EPISODES = 1_000

RUN_REWARD_SMOKE_TESTS = True
RUN_EXPERIMENT_3_3 = False
RUN_FINAL_VALIDATION = False
RUN_MATCHED_BEST_COMPARISON = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Best-parent checkpoint selection and isolated output paths

Experiment 3.3 accepts only the best-validation checkpoint as its parent. It does not fall back to `latest_checkpoint.pth`.

In [ ]:
if INITIALIZE_FROM != "best":
    raise ValueError(
        "Experiment 3.3 must initialize from Experiment 3.2 best, not latest."
    )

parent_checkpoints = CheckpointManager(INITIALIZE_FROM_EXPERIMENT)
experiment_3_3_checkpoints = CheckpointManager(EXPERIMENT_NAME)
initial_checkpoint = parent_checkpoints.best_checkpoint_path
experiment_3_3_best_checkpoint = (
    experiment_3_3_checkpoints.best_checkpoint_path
)
parent_checkpoint_available = initial_checkpoint.exists()

print("Required parent checkpoint:", initial_checkpoint.resolve())
print("Parent checkpoint available:", parent_checkpoint_available)
print("Experiment 3.3 output:", Path(EXPERIMENT_NAME).resolve())
print("  checkpoints:", experiment_3_3_checkpoints.checkpoint_directory)
print("  validation artifacts:", experiment_3_3_checkpoints.artifact_directory)
print("  metrics:", experiment_3_3_checkpoints.metrics_directory)

## Adaptive frontier-resolution reward smoke checks

These checks require no model checkpoint. They verify logarithmic growth with known-map size, inverse scaling with the number of reachable frontier regions, the cap, one-time resolution gating, and safe `F_t = 0` handling.

In [ ]:
adaptive_reward_defaults = ExplorationRewardConfig(
    adaptive_frontier_resolution_enabled=True,
    frontier_base_reward=0.5,
    frontier_map_scale=2.0,
    frontier_known_cell_scale=25.0,
    frontier_count_exponent=0.75,
    frontier_reward_cap=5.0,
)

if RUN_REWARD_SMOKE_TESTS:
    reward_small_map = compute_adaptive_frontier_resolution_reward(
        known_cells_before=25,
        reachable_frontier_count_before=6,
        resolved_frontier=True,
        config=adaptive_reward_defaults,
    )
    reward_large_map = compute_adaptive_frontier_resolution_reward(
        known_cells_before=250,
        reachable_frontier_count_before=6,
        resolved_frontier=True,
        config=adaptive_reward_defaults,
    )
    reward_few_frontiers = compute_adaptive_frontier_resolution_reward(
        known_cells_before=250,
        reachable_frontier_count_before=1,
        resolved_frontier=True,
        config=adaptive_reward_defaults,
    )
    reward_capped = compute_adaptive_frontier_resolution_reward(
        known_cells_before=1_000_000,
        reachable_frontier_count_before=1,
        resolved_frontier=True,
        config=adaptive_reward_defaults,
    )
    reward_repeated_observation = compute_adaptive_frontier_resolution_reward(
        known_cells_before=250,
        reachable_frontier_count_before=1,
        resolved_frontier=False,
        config=adaptive_reward_defaults,
    )
    reward_no_frontiers = compute_adaptive_frontier_resolution_reward(
        known_cells_before=250,
        reachable_frontier_count_before=0,
        resolved_frontier=True,
        config=adaptive_reward_defaults,
    )

    assert reward_small_map < reward_large_map < reward_few_frontiers
    assert reward_capped == adaptive_reward_defaults.frontier_reward_cap
    assert reward_repeated_observation == 0.0
    assert reward_no_frontiers == 0.0

    print("Known cells 25, frontiers 6:", reward_small_map)
    print("Known cells 250, frontiers 6:", reward_large_map)
    print("Known cells 250, frontiers 1:", reward_few_frontiers)
    print("Capped reward:", reward_capped)
    print("Repeated observation reward:", reward_repeated_observation)
    print("F_t = 0 reward:", reward_no_frontiers)
    print("Adaptive frontier-resolution reward checks passed.")

## Experiment 3.3 configuration

When the parent checkpoint exists, its complete Experiment 3.2 configuration is reconstructed. The only reward change is enabling/configuring adaptive frontier resolution. Epsilon is held at `0.02`; the inherited fine-tuning learning rate, architecture, mapper/tensors, all existing rewards, deadlocks, action mask, and validation schedule remain unchanged. `ContinuationConfig()` is deliberately disabled because this is a new experiment, not an in-place resume.

In [ ]:
experiment_3_3_config = None
parent_config = None

if parent_checkpoint_available:
    parent_config = MiniGridExplorationExperimentConfig.from_checkpoint(
        name="experiment_3_2_parent_reference",
        checkpoint_path=initial_checkpoint,
        output_directory=INITIALIZE_FROM_EXPERIMENT,
    )
    experiment_3_3_config = replace(
        parent_config,
        name=EXPERIMENT_NAME,
        initial_checkpoint=initial_checkpoint,
        reward=replace(
            parent_config.reward,
            adaptive_frontier_resolution_enabled=True,
            frontier_base_reward=0.5,
            frontier_map_scale=2.0,
            frontier_known_cell_scale=25.0,
            frontier_count_exponent=0.75,
            frontier_reward_cap=5.0,
        ),
        training=replace(
            parent_config.training,
            num_episodes=TRAINING_EPISODES,
            epsilon_start=0.02,
            epsilon_end=0.02,
        ),
        continuation=ContinuationConfig(),
        output=OutputConfig(EXPERIMENT_NAME),
    )

    print("Experiment name:", experiment_3_3_config.name)
    print("Initial checkpoint:", experiment_3_3_config.initial_checkpoint)
    print("Output directory:", experiment_3_3_config.output.directory)
    print("Training episodes:", experiment_3_3_config.training.num_episodes)
    print("Learning rate:", experiment_3_3_config.training.learning_rate)
    print("Epsilon:", experiment_3_3_config.training.epsilon_end)
    print("Adaptive reward:", asdict(experiment_3_3_config.reward))
else:
    print(
        "Configuration awaits the Experiment 3.2 best checkpoint. "
        "No latest-checkpoint fallback was used."
    )

## Initialization and scientific-identity checks

In [ ]:
experiment_3_3_trainer = None

if experiment_3_3_config is not None:
    # All non-reward behavioral components remain inherited.
    assert experiment_3_3_config.network == parent_config.network
    assert experiment_3_3_config.tensor == parent_config.tensor
    assert experiment_3_3_config.environment == parent_config.environment
    assert experiment_3_3_config.coverage == parent_config.coverage
    assert experiment_3_3_config.deadlock == parent_config.deadlock
    assert experiment_3_3_config.allowed_actions == parent_config.allowed_actions
    assert experiment_3_3_config.validation == parent_config.validation
    assert experiment_3_3_config.training.learning_rate == parent_config.training.learning_rate
    assert experiment_3_3_config.training.epsilon_end == 0.02
    assert not experiment_3_3_config.continuation.enabled
    assert experiment_3_3_config.output.directory == Path(EXPERIMENT_NAME)
    assert experiment_3_3_config.output.directory != parent_config.output.directory

    parent_reward = asdict(parent_config.reward)
    child_reward = asdict(experiment_3_3_config.reward)
    adaptive_reward_fields = {
        "adaptive_frontier_resolution_enabled",
        "frontier_base_reward",
        "frontier_map_scale",
        "frontier_known_cell_scale",
        "frontier_count_exponent",
        "frontier_reward_cap",
    }
    unchanged_reward_fields = set(parent_reward) - adaptive_reward_fields
    assert all(
        parent_reward[key] == child_reward[key]
        for key in unchanged_reward_fields
    )

    experiment_3_3_trainer = MiniGridExplorationTrainer(
        experiment_3_3_config, device
    ).initialize()
    assert experiment_3_3_trainer.source_checkpoint_path == initial_checkpoint
    assert len(experiment_3_3_trainer.replay_buffer) == 0
    assert len(experiment_3_3_trainer.optimizer.state) == 0
    assert experiment_3_3_trainer.training_history == []
    assert experiment_3_3_trainer.validation_history == []
    assert experiment_3_3_trainer.completed_episodes == 0
    assert experiment_3_3_trainer.total_steps == 0

    for policy_value, target_value in zip(
        experiment_3_3_trainer.policy_net.state_dict().values(),
        experiment_3_3_trainer.target_net.state_dict().values(),
    ):
        assert torch.equal(policy_value, target_value)

    lineage = experiment_3_3_trainer._metadata(0, 0.02, 0.0)
    assert lineage["parent_experiment"] == "experiment_3_2"
    assert lineage["initialization_checkpoint"] == (
        "best_validation_checkpoint.pth"
    )
    assert lineage["new_experiment_identity"]
    assert lineage["optimizer_initialization"] == "fresh_adam"
    assert lineage["target_initialization"] == (
        "hard_update_from_inherited_policy"
    )
    print("Best-policy inheritance and fresh Experiment 3.3 state verified.")
else:
    print("Initialization checks skipped until the parent best checkpoint exists.")

## Experiment 3.3 training launch

In [ ]:
if RUN_EXPERIMENT_3_3:
    if experiment_3_3_config is None:
        raise FileNotFoundError(
            "Experiment 3.2 best checkpoint is required: "
            f"{initial_checkpoint.resolve()}"
        )

    if experiment_3_3_trainer is None:
        experiment_3_3_trainer = MiniGridExplorationTrainer(
            experiment_3_3_config, device
        )

    experiment_3_3_policy, experiment_3_3_metadata = (
        experiment_3_3_trainer.train()
    )
    alert_training_complete("Experiment 3.3 fine-tuning finished.")
else:
    print("Ready. Set RUN_EXPERIMENT_3_3 = True to start training.")

## Best-checkpoint validation and result inspection

In [ ]:
if RUN_FINAL_VALIDATION:
    if experiment_3_3_config is None:
        raise FileNotFoundError(initial_checkpoint.resolve())
    if not experiment_3_3_best_checkpoint.exists():
        raise FileNotFoundError(experiment_3_3_best_checkpoint.resolve())

    experiment_3_3_best_policy, best_checkpoint = (
        CheckpointManager.load_policy(experiment_3_3_best_checkpoint, device)
    )
    evaluator = MiniGridExplorationEvaluator(experiment_3_3_config, device)
    best_results = evaluator.evaluate(
        experiment_3_3_best_policy,
        seeds=experiment_3_3_config.validation.large_validation_seeds,
        artifact_directory=(
            experiment_3_3_checkpoints.artifact_directory / "final_best"
        ),
    )
    best_summary = evaluator.summarize(best_results)
    print(best_summary)
    print("Lineage:", {
        "parent_experiment": best_checkpoint["parent_experiment"],
        "initialization_checkpoint": best_checkpoint["initialization_checkpoint"],
    })
else:
    print("Final validation remains disabled until Experiment 3.3 is trained.")

## Matched Experiment 3.2-best versus Experiment 3.3-best comparison

Both policies are evaluated greedily on the identical inherited 100-seed large-validation set. Comparison artifacts are written only under `experiment_3_3/`.

In [ ]:
if RUN_MATCHED_BEST_COMPARISON:
    if experiment_3_3_config is None:
        raise FileNotFoundError(initial_checkpoint.resolve())
    if not experiment_3_3_best_checkpoint.exists():
        raise FileNotFoundError(experiment_3_3_best_checkpoint.resolve())

    experiment_3_2_best_policy, _ = CheckpointManager.load_policy(
        initial_checkpoint, device
    )
    experiment_3_3_best_policy, _ = CheckpointManager.load_policy(
        experiment_3_3_best_checkpoint, device
    )
    evaluator = MiniGridExplorationEvaluator(experiment_3_3_config, device)
    comparison_seeds = (
        experiment_3_3_config.validation.large_validation_seeds
    )
    matched_results = {
        "Experiment 3.2 best": evaluator.evaluate(
            experiment_3_2_best_policy, seeds=comparison_seeds
        ),
        "Experiment 3.3 best": evaluator.evaluate(
            experiment_3_3_best_policy, seeds=comparison_seeds
        ),
    }
    comparison = save_comparison_artifacts(
        Path(EXPERIMENT_NAME) / "matched_parent_best_comparison",
        matched_results,
        thresholds=experiment_3_3_config.validation.coverage_thresholds,
    )
    print("Matched seeds:", (comparison_seeds[0], comparison_seeds[-1]))
    print(comparison)
else:
    print("Matched best-checkpoint comparison is disabled.")

## Saved metrics inspection

In [ ]:
best_metrics_path = experiment_3_3_checkpoints.metrics_directory / "best_validation.json"
training_metrics_path = experiment_3_3_checkpoints.metrics_directory / "training_metrics.json"

if best_metrics_path.exists():
    with best_metrics_path.open(encoding="utf-8") as file:
        saved_best_validation = json.load(file)
    print("Saved best validation:", saved_best_validation)

if training_metrics_path.exists():
    with training_metrics_path.open(encoding="utf-8") as file:
        saved_training_metrics = json.load(file)
    print("Saved training episodes:", len(saved_training_metrics))
    if saved_training_metrics:
        latest = saved_training_metrics[-1]
        print("Latest frontier resolution metrics:", {
            key: latest[key]
            for key in (
                "frontier_resolution_count",
                "total_frontier_resolution_reward",
                "mean_frontier_resolution_reward",
                "maximum_frontier_resolution_reward",
                "mean_known_cells_at_frontier_resolution",
                "mean_reachable_frontiers_at_resolution",
            )
        })
else:
    print("No Experiment 3.3 metrics have been written yet.")